# 01 · Negocio, datos y proceso analítico

**Edición docente · 8 de octubre de 2026 · Fundamentos de Business Analytics**

**Núcleo U1 / RA1 · CE 1.1–1.4 · 90–120 minutos más trabajo autónomo.**
Al finalizar podrás distinguir BA, BI y Big Data; formular una decisión; reconocer escalas de
medición; auditar calidad; describir CRISP-DM y discutir límites, sesgos y responsabilidad.

**Cómo trabajar:** ejecuta desde la primera celda. En cada simulador formula primero una predicción,
cambia un parámetro y justifica la diferencia. Los datos son casos docentes del repositorio o
simulaciones identificadas; no permiten inferir resultados de empresas reales.

Los controles requieren JupyterLab con `ipywidgets` y un kernel activo; las salidas guardadas permiten
leer los ejemplos sin ejecutar. Las funciones también pueden llamarse directamente, con lo que
el ejercicio sigue siendo utilizable si el visor no muestra widgets.

[Guía maestra](../guia_maestra_ba.html) · [Manual científico](../material_propio/FBA_manual_cientifico.pdf)

In [1]:
from pathlib import Path
import sys
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "_transversal" / "datos").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra Jupyter desde el repositorio completo o una subcarpeta.")
CURSO = RAIZ / "01_Fundamentos_Business_Analytics"
sys.path.insert(0, str(CURSO / "reproducibilidad"))
from fba_recursos import ventas, resumen_ventas, pregunta, DATOS, SEMILLA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .18})
pd.set_option("display.max_columns", 14)
df = ventas()
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · {len(df)} transacciones")

Python 3.12.13 · pandas 3.0.6 · 180 transacciones


<a id="negocio"></a>

## BA, BI y Big Data

BA conecta evidencia y decisión. BI organiza indicadores y reportes; puede aportar la base descriptiva
de un proceso BA. Big Data describe desafíos de volumen, velocidad y variedad que pueden requerir
arquitecturas distribuidas: un CSV de 180 ventas no es Big Data por tener muchas columnas.

Los cuatro tipos de análisis se distinguen por la pregunta, no por el lenguaje de programación:
describir, investigar explicaciones, anticipar un resultado y elegir una acción bajo restricciones.
Una segmentación diagnóstica sugiere hipótesis; para atribuir causas se necesita diseño y evidencia adicional.
Un análisis sencillo puede generar más valor que uno complejo si resuelve la decisión correcta.

In [2]:
tipos = pd.DataFrame([
    ["Descriptivo", "¿Cuánto vendimos?", "Ingresos por región", "Monto, Fecha, Region"],
    ["Diagnóstico", "¿Dónde se concentra una caída?", "Desagregar por producto y canal", "Ventas y contexto"],
    ["Predictivo", "¿Cuál será la demanda?", "Pronóstico con error explícito", "Historia temporal"],
    ["Prescriptivo", "¿Cómo asignar un presupuesto?", "Comparar acciones y restricciones", "Efectos, costos, capacidad"],
], columns=["Tipo", "Pregunta", "Evidencia", "Datos requeridos"])
display(tipos)
display(pd.Series(resumen_ventas(df), name="QuillayMarket · caso docente"))

,Tipo,Pregunta,Evidencia,Datos requeridos
0,Descriptivo,¿Cuánto vendimos?,Ingresos por región,"Monto, Fecha, Region"
1,Diagnóstico,¿Dónde se concentra una caída?,Desagregar por producto y canal,Ventas y contexto
2,Predictivo,¿Cuál será la demanda?,Pronóstico con error explícito,Historia temporal
3,Prescriptivo,¿Cómo asignar un presupuesto?,Comparar acciones y restricciones,"Efectos, costos, capacidad"


transacciones        1.800000e+02
unidades             4.730000e+02
ingreso              1.840223e+08
ticket_medio         1.022346e+06
precio_por_unidad    3.890534e+05
Name: QuillayMarket · caso docente, dtype: float64

### Una decisión antes del cálculo

RutaSur, empresa ficticia, considera una acción de retención. Suponemos que **sin actuar** se observa
una pérdida de 600 unidades monetarias por cliente que abandona. Si actuar reduce causalmente la
probabilidad de abandono de $p_0$ a $p_1$, cuesta $c$ y no produce otros efectos, la mejora esperada es
$\Delta V=(p_0-p_1)L-c$. Esta no es una predicción de ingresos realizada con QuillayMarket: es un escenario
separado para aprender a decidir. Las probabilidades de ambos escenarios tendrían que estimarse con
evidencia de intervención. Ser de alto riesgo no garantiza responder bien a una campaña.

In [3]:
def valor_accion(p0=.20, p1=.12, perdida=600., costo=30.):
    if not (0 <= p0 <= 1 and 0 <= p1 <= 1 and perdida >= 0 and costo >= 0):
        raise ValueError("Probabilidades entre 0 y 1 y montos no negativos")
    return (p0-p1)*perdida-costo

def explorar_decision(p0=.20, p1=.12, costo=30.):
    delta = valor_accion(p0, p1, costo=costo)
    print(f"Mejora esperada por cliente: {delta:.2f} u.m.")
    print("La acción supera a no actuar en este escenario." if delta > 0 else
          "La acción no supera a no actuar en este escenario.")
    print("Resultado condicionado a los supuestos de efecto causal, costo y pérdida.")
    return delta

explorar_decision()
widgets.interact(explorar_decision,
    p0=widgets.FloatSlider(value=.20,min=0,max=.5,step=.01,description="p sin acción"),
    p1=widgets.FloatSlider(value=.12,min=0,max=.5,step=.01,description="p con acción"),
    costo=widgets.FloatSlider(value=30,min=0,max=100,step=5));

Mejora esperada por cliente: 18.00 u.m.
La acción supera a no actuar en este escenario.
Resultado condicionado a los supuestos de efecto causal, costo y pérdida.


interactive(children=(FloatSlider(value=0.2, description='p sin acción', max=0.5, step=0.01), FloatSlider(valu…

<a id="tipos-datos"></a>

## Unidad de análisis, tipos y escalas

Una fila de QuillayMarket es una transacción, no un cliente. `ID_Venta` identifica filas y no debe
promediarse. `Region` y `Producto` son nominales; una evaluación bajo/medio/alto sería ordinal.
`Fecha` es temporal; las diferencias entre fechas son interpretables, sus cocientes no.
`Monto` y `Unidades` tienen cero sustantivo en este caso. Especifica siempre la moneda y el período.

Una variable almacenada como entero puede ser una categoría. Tipado informático y escala conceptual
son preguntas diferentes. Agregar registros antes de definir su grano puede duplicar ventas.

In [4]:
diccionario = pd.DataFrame([
 ["ID_Venta", "Identificador nominal", "Transacción", "Único y no nulo"],
 ["Fecha", "Fecha de calendario", "Día", "Fecha válida"],
 ["Region", "Categoría nominal", "Región", "Categoría documentada"],
 ["Unidades", "Conteo, razón", "Unidades vendidas", "Entero positivo"],
 ["Precio_Unitario", "Numérica, razón", "Moneda del caso por unidad", "No negativo"],
 ["Monto", "Numérica, razón", "Moneda del caso", "Unidades × Precio_Unitario"],
], columns=["Variable","Tipo conceptual","Unidad","Contrato"])
display(diccionario)
display(df.dtypes.rename("Tipo en pandas"))
assert df.ID_Venta.nunique() == len(df)
assert (df.Unidades > 0).all()
print("Grano y contrato de ingresos comprobados.")

,Variable,Tipo conceptual,Unidad,Contrato
0,ID_Venta,Identificador nominal,Transacción,Único y no nulo
1,Fecha,Fecha de calendario,Día,Fecha válida
2,Region,Categoría nominal,Región,Categoría documentada
3,Unidades,"Conteo, razón",Unidades vendidas,Entero positivo
4,Precio_Unitario,"Numérica, razón",Moneda del caso por unidad,No negativo
5,Monto,"Numérica, razón",Moneda del caso,Unidades × Precio_Unitario


ID_Venta                    int64
Fecha              datetime64[us]
Mes                           str
Region                        str
Producto                      str
Producto_Nombre               str
Vendedor                      str
Canal                         str
Unidades                    int64
Precio_Unitario           float64
Monto                     float64
Name: Tipo en pandas, dtype: object

Grano y contrato de ingresos comprobados.


<a id="calidad"></a>

## Calidad de datos y trazabilidad

La siguiente tabla defectuosa es una **copia de trabajo creada para este ejercicio**. Introducimos
un duplicado, un monto vacío y una categoría con espacios; el CSV permanece intacto.
Detectar no equivale a decidir: un registro repetido puede representar dos operaciones legítimas
si falta la clave, y eliminar ausencias puede sesgar el análisis. Se documenta cada transformación.
No imputaremos un ingreso por la media cuando puede reconstruirse a partir de cantidad y precio.

In [5]:
sucios = pd.concat([df.head(12), df.head(1)], ignore_index=True).copy()
sucios.loc[2, "Monto"] = np.nan
sucios.loc[3, "Region"] = "  " + sucios.loc[3, "Region"].lower() + "  "
def auditar(frame):
    return pd.Series({"filas":len(frame), "claves_repetidas":int(frame.ID_Venta.duplicated().sum()),
                      "montos_ausentes":int(frame.Monto.isna().sum()),
                      "categorias_region":frame.Region.nunique()})
display(auditar(sucios))
limpios = sucios.drop_duplicates("ID_Venta", keep="first").copy()
mapa = {r.strip().casefold(): r for r in df.Region.unique()}
limpios["Region"] = limpios.Region.str.strip().str.casefold().map(mapa)
faltan = limpios.Monto.isna()
limpios.loc[faltan,"Monto"] = limpios.loc[faltan,"Unidades"] * limpios.loc[faltan,"Precio_Unitario"]
display(auditar(limpios))
assert len(limpios) == 12 and limpios.Monto.notna().all()
assert np.allclose(limpios.Monto,df.head(12).Monto)
print("Bitácora: 1 copia retirada, 1 monto reconstruido por identidad contable, categorías normalizadas.")

filas                13
claves_repetidas      1
montos_ausentes       1
categorias_region     4
dtype: int64

filas                12
claves_repetidas      0
montos_ausentes       0
categorias_region     3
dtype: int64

Bitácora: 1 copia retirada, 1 monto reconstruido por identidad contable, categorías normalizadas.


<a id="crisp"></a>

## CRISP-DM aplicado y límites de la evidencia

El ciclo comprende negocio, datos, preparación, modelamiento, evaluación y despliegue. En Fundamentos
el «modelo» puede ser una tabla dinámica o una regla de decisión: no es obligatorio entrenar una red
neuronal. Volver a una fase anterior es normal. Una recomendación incluye responsable, horizonte,
indicador y condición de revisión. Su éxito se evalúa en la decisión, no solamente en el ajuste estadístico.

In [6]:
crisp = pd.DataFrame([
 ["Negocio", "Decidir qué segmento revisar", "Pregunta, alternativas y restricción"],
 ["Datos", "Definir transacción y horizonte", "Diccionario y cobertura temporal"],
 ["Preparación", "Validar claves y montos", "Bitácora de cambios"],
 ["Modelamiento", "Resumir por región/producto", "Tabla o modelo reproducible"],
 ["Evaluación", "Reconciliar totales y supuestos", "Resultado contrastado y limitaciones"],
 ["Despliegue", "Comunicar, asignar y monitorear", "Responsable, KPI y regla de revisión"],
],columns=["Fase","Aplicación","Evidencia entregable"])
display(crisp)

,Fase,Aplicación,Evidencia entregable
0,Negocio,Decidir qué segmento revisar,"Pregunta, alternativas y restricción"
1,Datos,Definir transacción y horizonte,Diccionario y cobertura temporal
2,Preparación,Validar claves y montos,Bitácora de cambios
3,Modelamiento,Resumir por región/producto,Tabla o modelo reproducible
4,Evaluación,Reconciliar totales y supuestos,Resultado contrastado y limitaciones
5,Despliegue,"Comunicar, asignar y monitorear","Responsable, KPI y regla de revisión"


<a id="etica"></a>

## Ética, sesgo y comunicación

El alcance ético incluye finalidad, minimización de datos, representatividad, explicación de decisiones
y posibilidad de revisión humana. Excluir una variable sensible no elimina las variables que actúan
como sustitutas. Una tabla anonimizada de forma superficial puede seguir permitiendo identificación
por combinaciones raras; esta práctica debe revisarse antes de reutilizar datos reales.

**Ejercicios con solución orientativa**

1. ¿El total de 180 registros equivale a 180 clientes? **No:** no hay identificador de cliente que lo pruebe.
2. Si $p_0=.20$, $p_1=.12$, $L=600$ y $c=30$, ¿conviene actuar? **Bajo esos supuestos**, $\Delta V=18$.
   Con costo 60 el valor es −12; cambia la decisión sin cambiar las probabilidades.
3. ¿Eliminar siempre un outlier mejora los datos? **No:** contrasta fuente, unidades y propósito.
4. Redacta una decisión de dos frases: evidencia verificable y acción propuesta. **Criterio de calidad:**
   separa hecho observado, hipótesis causal y condición que podría refutar la recomendación.

**Profundización docente:** reemplaza un efecto homogéneo por efectos distintos por segmento.
Explica por qué ordenar por probabilidad de abandono y ordenar por beneficio de intervención
pueden producir prioridades diferentes.

In [7]:
auto_01 = pregunta("¿Qué transforma una descripción en una decisión defendible?",
    ["Usar el algoritmo más complejo", "Definir alternativas, evidencia, costos y límites",
     "Mostrar muchas cifras decimales"],1,
    "Una recomendación necesita alternativas y supuestos; precisión numérica no sustituye pertinencia.")
assert np.isclose(valor_accion(.20,.12,600,30),18)
assert np.isclose(valor_accion(.20,.12,600,60),-12)
assert len(ventas()) == 180
print("Comprobaciones del laboratorio 01: OK")

Comprobaciones del laboratorio 01: OK


### Referencias y puente

Las fases siguen la documentación de [IBM sobre CRISP-DM](https://dataplatform.cloud.ibm.com/docs/content/wsd/data.html?context=cpdaas&locale=es).
Este laboratorio cubre RA1. La extensión sobre decisiones utiliza una formulación original de valor esperado
con supuestos explícitos. Continúa con el laboratorio 02 para describir patrones sin atribuir causalidad.